# Player Tracking Analysis - Zone Entry Method and Offensive Network Structure 
Data: [Big Data Cup 2021 data](https://github.com/bigdatacup/Big-Data-Cup-2021)


In [2]:
from scipy.spatial import Voronoi, voronoi_plot_2d
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.animation as animation
import seaborn as sns
import math

print("Libraries loaded successfully!")

Libraries loaded successfully!


# Understanding and Cleaning Data

In [4]:
df = pd.read_csv("hackathon_nwhl.csv")

In [5]:
df.head()

,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,...,Event,X Coordinate,Y Coordinate,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2
0,2021-01-23,Minnesota Whitecaps,Boston Pride,1,20:00,5,5,0,0,Boston Pride,...,Faceoff Win,100,43,Backhand,NaN,NaN,NaN,Stephanie Anderson,NaN,NaN
1,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:58,5,5,0,0,Boston Pride,...,Puck Recovery,107,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:57,5,5,0,0,Boston Pride,...,Zone Entry,125,28,Carried,NaN,NaN,NaN,Maddie Rowe,NaN,NaN
3,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:55,5,5,0,0,Boston Pride,...,Shot,131,28,Snapshot,On Net,t,f,NaN,NaN,NaN
4,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:53,5,5,0,0,Boston Pride,...,Faceoff Win,169,21,Backhand,NaN,NaN,NaN,Stephanie Anderson,NaN,NaN


In [6]:
df["game_date"].unique()

array(['2021-01-23', '2021-01-24', '2021-01-26', '2021-01-27',
       '2021-01-30', '2021-01-31', '2021-02-01'], dtype=object)

In [7]:
df[['Home Team', 'Away Team']].drop_duplicates()

,Home Team,Away Team
0,Minnesota Whitecaps,Boston Pride
1649,Buffalo Beauts,Connecticut Whale
3646,Metropolitan Riveters,Toronto Six
5365,Boston Pride,Buffalo Beauts
7163,Connecticut Whale,Metropolitan Riveters
9033,Toronto Six,Minnesota Whitecaps
10914,Toronto Six,Boston Pride
12644,Metropolitan Riveters,Minnesota Whitecaps
14461,Connecticut Whale,Boston Pride
16288,Buffalo Beauts,Toronto Six


In [8]:
df.columns.tolist()

['game_date',
 'Home Team',
 'Away Team',
 'Period',
 'Clock',
 'Home Team Skaters',
 'Away Team Skaters',
 'Home Team Goals',
 'Away Team Goals',
 'Team',
 'Player',
 'Event',
 'X Coordinate',
 'Y Coordinate',
 'Detail 1',
 'Detail 2',
 'Detail 3',
 'Detail 4',
 'Player 2',
 'X Coordinate 2',
 'Y Coordinate 2']

In [9]:
df['Event'].value_counts()

Event
Puck Recovery      8214
Play               7249
Incomplete Play    3430
Zone Entry         1944
Shot               1909
Dump In/Out        1863
Takeaway           1207
Faceoff Win         846
Penalty Taken       144
Goal                 76
Name: count, dtype: int64

In [10]:
df["Detail 1"].value_counts()

Detail 1
Direct                       6720
Indirect                     3959
Lost                         1637
Carried                      1267
Snapshot                      888
Wristshot                     718
Backhand                      618
Dumped                        578
Forehand                      227
Slapshot                      227
Retained                      226
Played                         99
Fan                            68
Deflection                     53
Tripping                       34
Wrap Around                    31
Interference                   29
Hooking                        21
Roughing                       19
Holding                        11
Cross-checking                  7
Slashing                        6
Too many men on the ice         4
Illegal Check to the Head       2
Game Misconduct                 2
High-sticking                   2
Unsportsmanlike conduct         2
Charging                        1
Feet                            1
Elbow

## Data Description

This dataset is a collection of National Women's Hockey League games over a seven day period, starting on January 23rd to February 1st. There are six teams that are present in this database: Minnesota Whitecaps, Boston Pride, Buffalo Beauts, Connecticut Whale, Metropolitan Riveters, Toronto Six. Each row represents different events that happen over the course of the game.

# Set up Dynamic Functionality + Global Variables
This database has lots of games, lots of teams all together. So first we want to filter down the data to one game. Then we want to look at each passing network inside of each period of play, so that we can better understand how the flow of the game changes over time. 

We have made this filtering dynamic so, that we can easy look at different games, periods and events (that we are making the networks on)

Global Variables

In [52]:
GOAL_X = 189, 42


Get Functions

In [73]:
def get_game_data(data):

    """
        requires: infromation about the game the user wishes to look at, specifically need a "Home Team" and "Date" col 
        return: a dataframe of data of that games home team and date
    """
    
    home_team = data.iloc[0]['Home Team']
    date = data.iloc[0]['game_date']
    
    df_game = df[(df['game_date'] == date) & (df['Home Team'] == home_team)].copy()
    return df_game

In [18]:
def get_period_events(game_data, period):
    
    """
        requires: a datafrane AND the period the user wishes to examine
        returns: a dataframe of that periods data
    """
    
    period_events = game_data[game_data['Period'] == period].copy()
    return period_events

In [19]:
def get_zone_entries(game_data, team):
    
    """ 
        requires: a dataframe AND team name the user wishes to examine
        returns: a set of all the zone entries that particular team made
    """
    
    zone_entries = game_data[(game_data['Team'] == team) & (game_data['Event'] == 'Zone Entry')].copy()
    return zone_entries

Math Functions

In [21]:
def euclidean_distance(p1,p2):
    return math.dist(p1, p2)

Network Centric Functions

In [108]:
def create_passing_networks(data, zone_entries):
    
    """
        requires: dataframe that includes puck events and team names AND dataframe of zone entries the user wishes to examine
        returns: a set of events that follow each zone entry in the zone_entries list
    """
    
    networks = []
    
    for possession_id, row in enumerate(zone_entries.iterrows()):
        
        _, row = row
        entry_id = row.name
        future = data[data.index > entry_id].copy()
        network_passes = []

        row = row.copy()
        row["possession_id"] = possession_id
        network_passes.append(row)
        
        for _, event in future.iterrows():
            event = event.copy()
            event["possession_id"] = possession_id
            network_passes.append(event)
            
            if event['Event'] == 'Faceoff Win':
                break
    
            if event['Team'] != row['Team']:
                break

        networks.append(network_passes)


    networks = pd.DataFrame([row for seq in networks for row in seq])


    return networks.copy()
                    

In [114]:
def create_network_obj(network_data, possession_id):
    
    """ uses: networkX to create a network which are made up of passes and shots.
        requires: a dataframe containing data to create the network data AND which possession that user wishes to examine
        returns: a Directed Networkx object identifying passing networks (and successful zone entries that lead to a goal and failures, takeaways etc.            
    """
    

    
    G = nx.DiGraph()

    

    
    

        
    pass

Misc. Functions

# Identifying Specific Passing Networks

We are going to start with looking at the first game in the dataset and start by looking at the home team zone entries.

In [41]:
first_game = df.iloc[[0]]

first_game

,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,...,Event,X Coordinate,Y Coordinate,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2
0,2021-01-23,Minnesota Whitecaps,Boston Pride,1,20:00,5,5,0,0,Boston Pride,...,Faceoff Win,100,43,Backhand,NaN,NaN,NaN,Stephanie Anderson,NaN,NaN


We are going to start by looking at the Minnesota Whitecaps, the home team

In [94]:
game_data = get_game_data(first_game)
first_period = get_period_events(game_data, 1)
mw_entries = get_zone_entries(first_period, "Minnesota Whitecaps")

In [96]:
mw_entries.head(5)

,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,...,Event,X Coordinate,Y Coordinate,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2
7,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:48,5,5,0,0,Minnesota Whitecaps,...,Zone Entry,125,6,Carried,NaN,NaN,NaN,Kaleigh Fratkin,NaN,NaN
19,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:23,5,5,0,0,Minnesota Whitecaps,...,Zone Entry,124,2,Carried,NaN,NaN,NaN,Kaleigh Fratkin,NaN,NaN
27,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:09,5,5,0,0,Minnesota Whitecaps,...,Zone Entry,124,20,Played,NaN,NaN,NaN,Taylor Turnquist,NaN,NaN
114,2021-01-23,Minnesota Whitecaps,Boston Pride,1,16:06,5,5,0,0,Minnesota Whitecaps,...,Zone Entry,124,81,Carried,NaN,NaN,NaN,Paige Capistran,NaN,NaN
131,2021-01-23,Minnesota Whitecaps,Boston Pride,1,15:40,5,5,0,0,Minnesota Whitecaps,...,Zone Entry,125,54,Carried,NaN,NaN,NaN,Paige Capistran,NaN,NaN


In [110]:
network = create_passing_networks(first_period, mw_entries)

In [112]:
network

,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,...,X Coordinate,Y Coordinate,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2,possession_id
7,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:48,5,5,0,0,Minnesota Whitecaps,...,125,6,Carried,NaN,NaN,NaN,Kaleigh Fratkin,NaN,NaN,0
8,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:47,5,5,0,0,Minnesota Whitecaps,...,157,0,Indirect,NaN,NaN,NaN,Meghan Lorence,198.0,38.0,0
9,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:43,5,5,0,0,Minnesota Whitecaps,...,193,20,Direct,NaN,NaN,NaN,Haley Mack,196.0,33.0,0
10,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:40,5,5,0,0,Boston Pride,...,3,49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
19,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:23,5,5,0,0,Minnesota Whitecaps,...,124,2,Carried,NaN,NaN,NaN,Kaleigh Fratkin,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511,2021-01-23,Minnesota Whitecaps,Boston Pride,1,0:54,5,4,1,1,Minnesota Whitecaps,...,176,16,Direct,NaN,NaN,NaN,Emma Stauber,127.0,42.0,14
512,2021-01-23,Minnesota Whitecaps,Boston Pride,1,0:52,5,4,1,1,Minnesota Whitecaps,...,127,46,Indirect,NaN,NaN,NaN,Maddie Rowe,158.0,78.0,14
513,2021-01-23,Minnesota Whitecaps,Boston Pride,1,0:50,5,4,1,1,Minnesota Whitecaps,...,158,78,Lost,NaN,NaN,NaN,NaN,NaN,NaN,14
514,2021-01-23,Minnesota Whitecaps,Boston Pride,1,0:49,5,4,1,1,Boston Pride,...,28,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14


In [106]:
first_possession = constuct_network_obj(network, 0)

NameError: name 'constuct_network_obj' is not defined